<a href="https://colab.research.google.com/github/ruslanmv/simple-ollama-environment/blob/master/simple_ollama_environment_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Simple Ollama Environment — Google Colab

This notebook recreates the **Simple Ollama Environment** (Python 3 + Jupyter + Ollama) in a Google Colab runtime.

It will:

1. Install the Python client **`ollama`**.
2. Download the **Ollama server** binary (Linux, no root required).
3. Start the Ollama server in the background inside this notebook.
4. Pull a small model (default: `qwen2.5:0.5b-instruct`).
5. Run a minimal chat example via the Python client.

> ⚠️ **Note:** Colab runtimes are ephemeral. If the runtime restarts, you need to rerun the setup cells.


## 0. (Optional) Enable GPU in Colab

Before running the setup cells, you can enable a GPU:

1. In the Colab menu, go to **Runtime → Change runtime type**.
2. Set **Hardware accelerator** to **GPU**.
3. Click **Save**.

Small models will also run on CPU, just more slowly.


In [1]:
import subprocess

print("🔎 Checking for NVIDIA GPU...")
try:
    gpu_info = subprocess.check_output(["nvidia-smi"], stderr=subprocess.STDOUT, text=True)
    print("✅ GPU detected! Summary:\n")
    # Show only first few lines
    print("\n".join(gpu_info.splitlines()[:8]))
except Exception:
    print("⚠️ No NVIDIA GPU detected or driver unavailable.")
    print("   Colab CPU-only runtimes can still run small models, just a bit slower.")


🔎 Checking for NVIDIA GPU...
✅ GPU detected! Summary:

Tue Nov 18 14:29:51 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


## 1. Install Python dependencies

We install the official **Ollama Python library** plus `requests` (for simple health checks).


In [2]:
!pip install -q "ollama>=0.6,<1.0" requests

import importlib
spec = importlib.util.find_spec("ollama")
print("✅ Installed 'ollama' Python client." if spec is not None else "❌ Failed to install 'ollama' client.")


✅ Installed 'ollama' Python client.


## 2. Download the Ollama server binary (no root)

We fetch the pre-built **Linux x86_64** Ollama binary directly into your home directory (no `sudo`, no systemd changes).

This follows the **non-root** installation pattern suggested in the Ollama community docs.


In [3]:
import os
import pathlib
import subprocess

HOME = os.path.expanduser("~")
OLLAMA_HOME = os.path.join(HOME, "opt", "ollama")
SRC_DIR = os.path.join(HOME, "src")

os.makedirs(OLLAMA_HOME, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

tar_path = os.path.join(SRC_DIR, "ollama-linux-amd64.tgz")
# Corrected path: the ollama binary is extracted into the 'bin' subdirectory
ollama_bin_path = os.path.join(OLLAMA_HOME, "bin", "ollama")

if not os.path.exists(ollama_bin_path):
    print("⬇️ Downloading Ollama Linux binary...")
    subprocess.run(
        [
            "curl",
            "-L",
            "https://ollama.com/download/ollama-linux-amd64.tgz",
            "-o",
            tar_path,
        ],
        check=True,
    )

    # Verify downloaded file size. Ollama binary is usually several MBs.
    if not os.path.exists(tar_path) or os.path.getsize(tar_path) < 1024 * 1024: # Less than 1MB is suspicious
        raise RuntimeError(f"Downloaded Ollama tarball is missing or too small: {tar_path}")

    print("📦 Extracting...")
    # The tar command extracts contents directly into OLLAMA_HOME, but the binary is in a 'bin' subdir
    subprocess.run(
        ["tar", "-C", OLLAMA_HOME, "-xzf", tar_path],
        check=True,
    )

    # Verify existence immediately after extraction with the corrected path
    if not os.path.exists(ollama_bin_path):
        print(f"⚠️ Ollama binary not found after extraction. Listing contents of {OLLAMA_HOME} recursively for diagnosis:")
        ls_output = subprocess.check_output(["ls", "-laR", OLLAMA_HOME], text=True, stderr=subprocess.STDOUT)
        print(ls_output) # Print the captured output
        raise RuntimeError(f"Ollama binary not found at {ollama_bin_path} after extraction. See directory listing above.")
else:
    print("✅ Ollama binary already present, skipping download.")

# Final check (redundant if previous checks are robust, but good for safety)
if not os.path.exists(ollama_bin_path):
    raise RuntimeError(f"Ollama binary not found at {ollama_bin_path}")

print("✅ Ollama binary ready at:", ollama_bin_path)

⬇️ Downloading Ollama Linux binary...
📦 Extracting...
✅ Ollama binary ready at: /root/opt/ollama/bin/ollama


## 3. Start the Ollama server in the background

This cell:

- Configures some basic environment variables.
- Starts `ollama serve` as a **background process** inside the Python kernel.
- Waits until the HTTP API is healthy at `http://127.0.0.1:11434`.


In [4]:
import os
import time
import subprocess
import requests

HOME = os.path.expanduser("~")
OLLAMA_HOME = os.path.join(HOME, "opt", "ollama")
# Corrected OLLAMA_BIN path to point to the 'bin' subdirectory
OLLAMA_BIN = os.path.join(OLLAMA_HOME, "bin", "ollama")

if not os.path.exists(OLLAMA_BIN):
    raise RuntimeError("Ollama binary not found. Run the previous install cell first.")

# Stop an existing server (if any) from a previous run
try:
    OLLAMA_SERVER_PROCESS  # type: ignore[name-defined]
    if OLLAMA_SERVER_PROCESS.poll() is None:
        print("🛑 Stopping existing Ollama server...")
        OLLAMA_SERVER_PROCESS.terminate()
        try:
            OLLAMA_SERVER_PROCESS.wait(timeout=10)
        except subprocess.TimeoutExpired:
            OLLAMA_SERVER_PROCESS.kill()
except NameError:
    pass

# Where to store models (in the Colab filesystem)
os.environ.setdefault("OLLAMA_MODELS", os.path.join("/content", "ollama-models"))
os.makedirs(os.environ["OLLAMA_MODELS"], exist_ok=True)

# Basic tuning flags
os.environ.setdefault("OLLAMA_MAX_LOADED_MODELS", "1")
os.environ.setdefault("OLLAMA_NUM_PARALLEL", "1")

# Host where the API will be available
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

print("🚀 Starting Ollama server...")
OLLAMA_SERVER_PROCESS = subprocess.Popen(
    [OLLAMA_BIN, "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

# Wait for health endpoint
for i in range(60):
    try:
        r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        if r.ok:
            print("✅ Ollama server is up at http://127.0.0.1:11434")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama server did not become healthy in time.")

🚀 Starting Ollama server...
✅ Ollama server is up at http://127.0.0.1:11434


## 4. Pull a small model

We'll pull the same small instruction-tuned model used in the original **Simple Ollama Environment**:

- Default: `qwen2.5:0.5b-instruct`

You can change the model name before running the cell (e.g. to `llama3.2:1b` or `qwen3:0.6b`, if available on Ollama).


In [12]:
import os
import subprocess
# Change this if you want a different model
MODEL_NAME = "qwen2.5:0.5b-instruct"
HOME = os.path.expanduser("~")
OLLAMA_HOME = os.path.join(HOME, "opt", "ollama")
# Corrected OLLAMA_BIN path to point to the 'bin' subdirectory
OLLAMA_BIN = os.path.join(OLLAMA_HOME, "bin", "ollama")
print(f":inbox_tray: Pulling model: {MODEL_NAME}")
try:
    result = subprocess.run([OLLAMA_BIN, "pull", MODEL_NAME], check=True, capture_output=True, text=True)
    print(result.stdout)
    print(":white_check_mark: Model pulled and ready.")
except subprocess.CalledProcessError as e:
    print(f":x: Failed to pull model: {MODEL_NAME}")
    print(f"Error code: {e.returncode}")
    print(f"Output: {e.stdout}")
    print(f"Error output: {e.stderr}")
    raise

:inbox_tray: Pulling model: qwen2.5:0.5b-instruct

:white_check_mark: Model pulled and ready.


## 5. Run a quick chat from Python

Now that:

- the **Ollama server** is running, and  
- a model is pulled,

we can use the `ollama` Python client to send a simple chat request.


In [13]:
import ollama
import os

# Ensure the client talks to our local server
os.environ.setdefault("OLLAMA_HOST", "http://127.0.0.1:11434")

MODEL_NAME = "qwen2.5:0.5b-instruct"  # keep in sync with the pull cell above

print("✅ Using Ollama Python client version:", getattr(ollama, "__version__", "unknown"))
print("🧠 Talking to model:", MODEL_NAME)

prompt = "Di' solo 'Ciao!' in italiano e poi dammi 1 consiglio per studiare meglio."

resp = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
)

print("\n💬 User:")
print(prompt)
print("\n🤖 Model reply:\n")
print(resp["message"]["content"])


✅ Using Ollama Python client version: unknown
🧠 Talking to model: qwen2.5:0.5b-instruct

💬 User:
Di' solo 'Ciao!' in italiano e poi dammi 1 consiglio per studiare meglio.

🤖 Model reply:

"Ciao! Ecco un semplice "Ciao" in italiano: Ciao!" È una frase famosa che può essere ben usata e molto amica per chiunque, anche se non è comune o preferibile.

Un consiglio per studiare meglio è sempre di esaminare l'accento sulle parole da cui viene eseguito il "Ciao!". Certo, alcuni elementi del linguaggio come "ma", "è", e "sai" possono variarsi in terminologia. Tuttavia, un consiglio più sottile per studiare meglio è di farlo spesso parlare con altre persone o di scrivere un po' del testo per vedere se si capisce facilmente.

Inoltre, per migliorare le tue capacità di comunicazione, ti consiglio di fare i segni di luce al tuo pensiero e rispondere a domande con certezza. Non solo questo, ma anche di fare l'analisi delle tue scuole o le lezioni che hai ottenuto per avviare il proprio progresso.

N

## 6. Next steps

You now have:

- An **Ollama server** running inside Colab.
- A **Python client** ready to call `ollama.chat` / `ollama.generate`.
- At least one **local model** pulled.

From here you can:

- Swap models (edit `MODEL_NAME` and re-run the pull + chat cells).
- Use `stream=True` in `ollama.chat` for streaming responses.
- Integrate Ollama with other libraries (LangChain, LlamaIndex, etc.) in additional cells.

Happy hacking! 🚀
